In [4]:
import sys
print("python:", sys.version.split()[0])
print("kernel:", sys.executable, "\n")

for name in ["coffea", "uproot", "awkward", "numpy", "matplotlib", "hist", "mplhep", "vector"]:
    try:
        mod = __import__(name)
        print(f"  [ok]      {name:<12s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  [MISSING] {name}")

python: 3.10.14
kernel: /depot/cms/conda_envs/bhanda25/Coffea-Santosh/bin/python 

  [ok]      coffea       2024.2.2
  [ok]      uproot       5.3.10
  [ok]      awkward      2.6.5
  [ok]      numpy        1.26.4
  [ok]      matplotlib   3.9.1
  [ok]      hist         2.7.3
  [ok]      mplhep       0.3.50
  [ok]      vector       1.4.1


In [5]:
import os
import pathlib
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

# coffea 2024.2.2 warns on import that a future release will hand its four-vector
# maths to `vector`, the package you used in Lab 1 (more on that in §1.3). Nothing
# we do is affected, so we silence that one warning. Silencing warnings you have
# not read is a bad habit; this one has been read.
warnings.filterwarnings("ignore", category=FutureWarning, module="coffea.*")

import awkward as ak
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

import coffea
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, BaseSchema
from coffea.nanoevents.methods import candidate

from utils import chunk_jobs, plot_dimuon_spectrum

hep.style.use("CMS")
print("coffea", coffea.__version__)

coffea 2024.2.2


In [6]:
import json
import urllib.request

RECID = 75589
with urllib.request.urlopen(f"https://opendata.cern.ch/api/records/{RECID}", timeout=60) as r:
    meta = json.load(r)["metadata"]

print("title      :", meta["title"])
print("DOI        :", meta["doi"])
print("licence    :", meta["license"])
print("distribution:", json.dumps(meta["distribution"], indent=2))
meta_files = []
for i in range(5):
    for j in range(len(meta["_file_indices"][i]['files'])):
        #print("Files:", meta["_file_indices"][i]['files'][j])
        meta_files.append(meta["_file_indices"][i]['files'][j])
print(meta_files[0:3])

title      : /ZZTo4L_TuneCP5_13TeV_powheg_pythia8/RunIISummer20UL16NanoAODv9-106X_mcRun2_asymptotic_v17-v1/NANOAODSIM
DOI        : 10.7483/OPENDATA.CMS.3Z2W.O02W
licence    : {'attribution': 'CC0-1.0'}
distribution: {
  "formats": [
    "nanoaodsim",
    "root"
  ],
  "number_events": 52104000,
  "number_files": 99,
  "size": 71479443458
}
[{'availability': 'online', 'bucket': 'db3870f1-c425-4e5e-83a3-6350604fa413', 'checksum': 'adler32:6af0d28f', 'file_id': '227c3166-3a84-4c93-b561-9e60463cc0c4', 'filename': '051E9D22-4F30-8E49-8477-644E65768282.root', 'key': 'CMS_mc_RunIISummer20UL16NanoAODv9_ZZTo4L_TuneCP5_13TeV_powheg_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v1_2430000_file_index.json_0', 'size': 232295116, 'tags': {}, 'uri': 'root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/ZZTo4L_TuneCP5_13TeV_powheg_pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v1/2430000/051E9D22-4F30-8E49-8477-644E65768282.root', 'version_id': '3296448b-1ed8-401b-884c-f0256bdc4ea7'

In [ ]:
# !!! ONLY RUN THIS 1 TIME EVER UNLESS YOU WANT TO WAIT 1 HOUR TO DO ANYTHING !!! #
import os, pathlib, shutil, subprocess, time
from XRootD import client
import urllib.request

for i in range(len(meta_files)):
    XROOTD_URL = meta_files[i]["uri"]
    HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
    EXPECTED   = meta_files[i]["size"]
    
    DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_background_data"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE = DATA_DIR / meta_files[i]["key"]
    
    if LOCAL_FILE.exists() and LOCAL_FILE.stat().st_size == EXPECTED:
        print("Already downloaded:", LOCAL_FILE)
    else:
        t0, ok = time.time(), False
    
        if shutil.which("xrdcp"):
            print("Trying xrdcp ...")
            ok = subprocess.run(["xrdcp", "-f", XROOTD_URL, str(LOCAL_FILE)]).returncode == 0
    
        if not ok:
            try:
                print("Trying XRootD python bindings ...")
                ok = client.FileSystem("root://eospublic.cern.ch").copy(
                         XROOTD_URL, str(LOCAL_FILE), force=True)[0].ok
            except ImportError:
                pass
    
        if not ok:
            print("Falling back to HTTPS ...")
            urllib.request.urlretrieve(HTTPS_URL, LOCAL_FILE)
    
        print(f"done in {time.time()-t0:.0f} s -> {LOCAL_FILE}")
    
    print(f"\nsize on disk: {LOCAL_FILE.stat().st_size:,} bytes")
    print("Loops:", i)

In [ ]:
import zlib
for i in range(5):
    for j in range(len(meta["_file_indices"][i]['files'])):
        # YOUR CODE HERE
        XROOTD_URL = meta_files[i]["uri"]
        HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
        EXPECTED   = meta_files[i]["size"]
        
        DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_background_data"
        # DATA_DIR.mkdir(parents=True, exist_ok=True)
        LOCAL_FILE = DATA_DIR / meta["_file_indices"][i]["files"][j]["key"]
                  
        local_size = LOCAL_FILE.stat().st_size
        print (f"Filesize (bytes): {local_size:,}")
        print ("Filesize (MB): " + str(round(local_size/1000000, 2)))
        print("Size matches:", local_size == meta["_file_indices"][i]["files"][j]["size"])
        
        running_value = 1
        with open(LOCAL_FILE, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                running_value = zlib.adler32(chunk, running_value)
        checksum = f"{running_value:08x}"
        expected_checksum = meta["_file_indices"][i]["files"][j]["checksum"].replace("adler32:", "")
        print("Checksum matches: " + str(checksum == expected_checksum))

In [7]:
import pathlib
import uproot

events = []
for i in range(5):
    for j in range(len(meta["_file_indices"][i]['files'])):
        # YOUR CODE HERE
        XROOTD_URL = meta_files[i]["uri"]
        HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
        EXPECTED   = meta_files[i]["size"]
        
        DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_background_data"
        # DATA_DIR.mkdir(parents=True, exist_ok=True)
        LOCAL_FILE = DATA_DIR / meta["_file_indices"][i]["files"][j]["key"]
        
        try:
            FILE = str(LOCAL_FILE)                      # from Part 2
            assert pathlib.Path(FILE).exists()
            print("Using local file from Part 2")
        except (NameError, AssertionError):
            FILE = ("root://eospublic.cern.ch//eos/opendata/cms/derived-data/"
                    "AOD2NanoAODOutreachTool/ForHiggsTo4Leptons/ZZTo4mu.root")
            # /eos/opendata/cms/dataset-semantics/NanoAODSIM/75589/ZZTo4L_TuneCP5_13TeV_powheg_pythia8_doc.root
            print("Local file not found - streaming over XRootD (slower)")
        
        events.append(uproot.open(FILE)["Events"])
count = 0
for i in range(len(events)):
    count += events[i].num_entries
#print(f"\nnumber of events: {(events[0].num_entries)}")
print("Number of Events:", count)

Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local fi

In [ ]:
for i in range(1):
    for name, branch in events[0].items():
        print(f"  {name:<24} {branch.typename}")

In [9]:
import numpy as np
import awkward as ak
import uproot
import vector

vector.register_awkward()      # teaches awkward about Lorentz vectors - do this ONCE

print("uproot ", uproot.__version__)
print("awkward", ak.__version__)
print("vector ", vector.__version__)

uproot  5.3.10
awkward 2.6.5
vector  1.4.1


In [8]:
for i in range(len(events)):
    peek = events[i].arrays(["nMuon", "Muon_pt", "Muon_charge"], entry_stop=9)

for i in range(9):
    pts     = [f"{p:.1f}" for p in peek.Muon_pt[i].to_list()]
    charges = [f"{q:+d}"  for q in peek.Muon_charge[i].to_list()]
    print(f"event {i}: nMuon={peek.nMuon[i]}  pt=[{', '.join(pts)}]  charge=[{', '.join(charges)}]")

event 0: nMuon=2  pt=[7.6, 4.3]  charge=[-1, +1]
event 1: nMuon=1  pt=[7.2]  charge=[-1]
event 2: nMuon=2  pt=[4.7, 3.7]  charge=[-1, +1]
event 3: nMuon=2  pt=[29.4, 28.5]  charge=[+1, -1]
event 4: nMuon=1  pt=[5.7]  charge=[-1]
event 5: nMuon=0  pt=[]  charge=[]
event 6: nMuon=2  pt=[26.0, 3.6]  charge=[+1, -1]
event 7: nMuon=0  pt=[]  charge=[]
event 8: nMuon=1  pt=[14.9]  charge=[+1]


In [10]:
BRANCHES = ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", "Muon_charge",
            "Muon_pfRelIso04_all", "Muon_dxy", "Muon_dxyErr", "Muon_dz", "Muon_dzErr"]

N_EVENTS = None         # while developing set this to e.g. 200_000; None = all events
for i in range(len(events)):
    raw = events[i].arrays(BRANCHES, entry_stop=N_EVENTS)
    print(f"read {len(raw):,} events, {raw.nbytes/1e6:.0f} MB in memory")

read 168,000 events, 19 MB in memory
read 84,000 events, 10 MB in memory
read 1,008,000 events, 117 MB in memory
read 1,383,000 events, 160 MB in memory
read 84,000 events, 10 MB in memory
read 1,008,000 events, 117 MB in memory
read 168,000 events, 19 MB in memory
read 12,000 events, 1 MB in memory
read 9,000 events, 1 MB in memory
read 924,000 events, 107 MB in memory
read 3,000 events, 0 MB in memory
read 1,362,000 events, 158 MB in memory
read 8,000 events, 1 MB in memory
read 980,000 events, 113 MB in memory
read 1,023,000 events, 118 MB in memory
read 982,000 events, 114 MB in memory
read 22,000 events, 3 MB in memory
read 12,000 events, 1 MB in memory
read 504,000 events, 58 MB in memory
read 1,008,000 events, 117 MB in memory
read 84,000 events, 10 MB in memory
read 9,000 events, 1 MB in memory
read 75,000 events, 9 MB in memory
read 1,484,000 events, 172 MB in memory
read 92,000 events, 11 MB in memory
read 2,000 events, 0 MB in memory
read 1,043,000 events, 121 MB in memory
r

In [11]:
type(raw)

awkward.highlevel.Array

In [12]:
pt = raw.Muon_pt
print(f"pt: {pt.type}")

# --- 1. ak.num : how many objects per event ------------------------------
print("1) ak.num(pt)[:10] =", ak.num(pt)[:10].to_list())
print("   mean muons/event = %.2f, max = %d" % (ak.mean(ak.num(pt)), ak.max(ak.num(pt))))

pt: 871000 * var * float32
1) ak.num(pt)[:10] = [2, 1, 2, 2, 1, 0, 2, 0, 1, 0]
   mean muons/event = 0.89, max = 9
